# Detector event matching (sel_all only)

Match events **before any selection** across detector-variation samples by
`(E / nuE, run, subrun, evt)` and write per-file `*_matched.df`.

Always use **`sel_all`** inputs (raw `evt` / `trk` / `hdr`). Do **not** match at
`sel_mup` / `sel_2prong` — that intersects post-selection survivors and drops
differential efficiency.

Backends: `dent_match_common_events.py` (sel_all), optional SCE script.

Downstream notebooks walk the selection pipeline on these matched files:
`wiremod.ipynb`, `dent.ipynb` → `systematics-detector.ipynb`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE, SPRING_GEN1_ROOT
from analysis_village.numucc_1p0pi.syst_detvar_common import log, run_dent_match

DFS = Path(os.environ.get("NUMUCC_SPRING_GEN1_ROOT", SPRING_GEN1_ROOT))
OUT_CACHE = Path(os.environ.get(
    "DETECTOR_MATCH_CACHE",
    str(PLOTS_BASE / "systematics-final" / "DetectorMatch" / "cache"),
))
OUT_CACHE.mkdir(parents=True, exist_ok=True)

RUN_WIREMOD_MATCH = True
RUN_DENT_MATCH = True
MAX_FILES = None  # e.g. 3 for a smoke test

print("DFS =", DFS)
print("OUT_CACHE =", OUT_CACHE)


## WireMod match (`sel_all` + calo)

Intersect YZ / XTXW / CV event keys at **`sel_all`**, then write `*_matched.df`.

**YZ / XTXW** are **sel_all + updatecalo** (`evt_cv` / calo ± / `evt_efield` + `trk`).
Matching preserves those tables for the WireMod envelope walk.

**CV** is `2026_09_04_172912__sel_all-mc-CV` (plain nominal) — match / POT reference only.


In [ ]:
# sel_all WireMod / CV dirs (override as needed).
# YZ/XTXW: updatecalo (cv + calo ± + efield). CV: plain nominal (match + POT only).
WIREMOD_VARIATIONS = {
    "yz": str(
        DFS
        / os.environ.get(
            "WIREMOD_YZ_SEL_ALL", "2026_09_14_025216__sel_all-mc-BNB_cosmics-WireModYZ"
        )
    ),
    "xtxw": str(
        DFS
        / os.environ.get(
            "WIREMOD_XTXW_SEL_ALL", "2026_09_14_024629__sel_all-mc-BNB_cosmics-WireModXTXW"
        )
    ),
    "cv": str(DFS / os.environ.get("WIREMOD_CV_SEL_ALL", "2026_09_04_172912__sel_all-mc-CV")),
}

WIREMOD_VARIATIONS = {k: v for k, v in WIREMOD_VARIATIONS.items() if Path(v).is_dir()}
print("WireMod sel_all dirs:", WIREMOD_VARIATIONS)

if RUN_WIREMOD_MATCH and len(WIREMOD_VARIATIONS) >= 2:
    log("WireMod sel_all match …")
    rc = run_dent_match(
        WIREMOD_VARIATIONS,
        fmt="sel_all",
        filename_str="sel_all",
        summary_csv=str(OUT_CACHE / "wiremod_matched_summary-sel_all.csv"),
        max_files=MAX_FILES,
    )
    print("wiremod match exit:", rc)
elif RUN_WIREMOD_MATCH:
    print("skip WireMod match: need ≥2 existing sel_all variation dirs")
else:
    print("skip WireMod match")


## DENT match (`sel_all`)

Same key definition (`hdr` + generator `evt.mc.E`).

**Pair-wise matching:** when CV/DENT are produced in batches that can share
`(run, subrun, evt, E)` across batches, match **within each (CVᵢ, DENTᵢ) pair**
separately, then write all matched files under the same
`matched/{cv,dent}/` trees (filenames differ by campaign tag). Downstream
`dent.ipynb` treats the union of CV matched files as CV and DENT as DENT.


In [ ]:
# Match each (CV, DENT) pair separately so batch metadata overlap cannot
# cross-match CV1↔DENT2 (etc.). Outputs share matched/{cv,dent}/ (basenames differ).
DENT_PAIRS = [
    {
        "name": "pair1",
        "cv": DFS / "2026_09_14_142556__sel_all-mc-CV1",
        "dent": DFS / "2026_09_14_142722__sel_all-mc-DENT1",
    },
    {
        "name": "pair2",
        "cv": DFS / "2026_09_14_153348__sel_all-mc-CV2",
        "dent": DFS / "2026_09_14_142928__sel_all-mc-DENT2",
    },
]

DENT_MATCHED_OUT = Path(PLOTS_BASE) / "systematics-final" / "DENT-highstats" / "matched"
DENT_MATCHED_SUFFIX = "_matched_hs"
DENT_CACHE = Path(PLOTS_BASE) / "systematics-final" / "DENT-highstats" / "cache"
DENT_N_WORKERS = int(os.environ.get("DENT_MATCH_N_WORKERS", "8"))
DENT_FILE_TIMEOUT = float(os.environ.get("DENT_MATCH_FILE_TIMEOUT", "600"))
DENT_META_RETRIES = int(os.environ.get("DENT_MATCH_META_RETRIES", "3"))

print("DENT pairs:")
for p in DENT_PAIRS:
    print(f"  {p['name']}: cv={p['cv'].name}  dent={p['dent'].name}")
print("matched out:", DENT_MATCHED_OUT, "suffix:", DENT_MATCHED_SUFFIX)

if RUN_DENT_MATCH:
    DENT_MATCHED_OUT.mkdir(parents=True, exist_ok=True)
    DENT_CACHE.mkdir(parents=True, exist_ok=True)
    pair_key_sets = []
    for p in DENT_PAIRS:
        if not p["cv"].is_dir() or not p["dent"].is_dir():
            raise FileNotFoundError(f"missing pair dirs for {p['name']}: {p['cv']} / {p['dent']}")
        keys_pkl = DENT_CACHE / f"dent_common_keys_sel_all_{p['name']}.pkl"
        log(f"DENT sel_all match [{p['name']}] …")
        rc = run_dent_match(
            {"cv": str(p["cv"]), "dent": str(p["dent"])},
            fmt="sel_all",
            filename_str="sel_all",
            summary_csv=str(OUT_CACHE / f"dent_matched_summary-sel_all-{p['name']}.csv"),
            matched_out_dir=str(DENT_MATCHED_OUT),
            matched_suffix=DENT_MATCHED_SUFFIX,
            common_keys_pkl=str(keys_pkl),
            n_workers=DENT_N_WORKERS,
            file_timeout=DENT_FILE_TIMEOUT,
            meta_retries=DENT_META_RETRIES,
            max_files=MAX_FILES,
        )
        print(f"dent match [{p['name']}] exit:", rc)
        if keys_pkl.is_file():
            import pickle

            with open(keys_pkl, "rb") as fh:
                pair_key_sets.append(set(pickle.load(fh)))
    if pair_key_sets:
        import pickle

        union_keys = set.union(*pair_key_sets)
        union_pkl = DENT_CACHE / "dent_common_keys_sel_all.pkl"
        with open(union_pkl, "wb") as fh:
            pickle.dump(union_keys, fh, protocol=pickle.HIGHEST_PROTOCOL)
        print(
            f"wrote union common keys ({len(union_keys):,}) ← "
            + " + ".join(f"{len(s):,}" for s in pair_key_sets)
            + f" → {union_pkl}"
        )
else:
    print("skip DENT match")


## Optional SCE

Still `scripts/sce_match_common_events.py` — prefer sel_all inputs there too.
Detector Product B uses **WireMod + DENT**, not SCE.


In [ ]:
print("Matching complete (sel_all). Next: wiremod.ipynb / dent.ipynb (pipeline walk on matched files).")
